# 02 - Exploratory Data Analysis (EDA) & Product Demand Analysis

## Objectives
1. Inspect the processed daily product-level demand series (`data/processed/daily_product_demand.parquet`).
2. Analyze total daily volume, distribution skewness, and intermittent zero-demand rates.
3. Investigate product catalog history length and sparsity tiers.
4. Inspect day-of-week and monthly seasonal patterns (noting the sample covers ~1 calendar year).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Load processed daily product demand
data_path = Path("../data/processed/daily_product_demand.parquet")
df = pd.read_parquet(data_path)
df["date"] = pd.to_datetime(df["date"])

print(f"Loaded shape: {df.shape[0]:,} rows across {df['product_id'].nunique():,} unique products.")
print(f"Date span: {df['date'].min().date()} to {df['date'].max().date()} ({df['date'].nunique()} days).")
df.head()

### 1. Total Daily Demand Trend
- Demand shows strong weekly seasonality and an overall upward surge toward Q4 (holiday peak in Oct-Nov 2011).
- Visible in `outputs/figures/01_total_daily_demand.png`.

In [ ]:
daily_total = df.groupby("date")["demand"].sum().reset_index()
daily_total["ma_7"] = daily_total["demand"].rolling(7, min_periods=1).mean()

plt.figure(figsize=(12, 4.5))
plt.plot(daily_total["date"], daily_total["demand"], alpha=0.5, label="Daily Total")
plt.plot(daily_total["date"], daily_total["ma_7"], color="#2563EB", linewidth=2, label="7-Day Moving Average")
plt.title("Total Daily Demand Across All Products")
plt.xlabel("Date")
plt.ylabel("Units Sold")
plt.legend()
plt.tight_layout()
plt.show()

### 2. Product-Level Demand Distribution
- Highly right-skewed: most product-days feature 0 to 10 units, with occasional wholesale bulk orders exceeding hundreds of units.
- Log-transformation $\log(1 + \text{demand})$ normalizes the distribution for evaluation.

In [ ]:
pos_demand = df[df["demand"] > 0]["demand"]
print("Positive demand summary:")
print(pos_demand.describe())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.hist(pos_demand, bins=40, range=(0, 100), color="#3B82F6", edgecolor="white")
ax1.set_title("Demand Distribution (0-100 units)")
ax2.hist(np.log1p(pos_demand), bins=40, color="#10B981", edgecolor="white")
ax2.set_title("Log-Transformed: log(1 + demand)")
plt.tight_layout()
plt.show()

### 3. Day of Week Patterns
- Thursday and Tuesday exhibit highest average demand volume.
- Saturdays in the UK wholesale operations show virtually zero transactions (closed or minimal operations).

In [ ]:
df["dow"] = df["date"].dt.day_name()
order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
dow_avg = df.groupby("dow")["demand"].mean().reindex(order)

plt.figure(figsize=(8, 4))
dow_avg.plot(kind="bar", color="#0284C7", edgecolor="white")
plt.title("Average Daily Demand per Product by Day of Week")
plt.ylabel("Mean Units per Product-Day")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### 4. Product Catalog History Length & Sparsity Tiers
- Out of 3,804 products, 2,876 (75.6%) span > 180 calendar days.
- 230 products have fewer than 28 calendar days of history and cannot support 28-day lag features.

In [ ]:
from retail_forecasting.features.build_features import analyze_product_histories
diag_df, summary = analyze_product_histories(df)
print("Catalog History Tiers:")
for tier, cnt in summary["history_tiers"].items():
    print(f"  {tier}: {cnt} products ({cnt / len(diag_df) * 100:.1f}%)")